In this section we will be revisiting Countmin Sketch and constructing *range queries*. That is, we want to efficiently estimate counts between arbitrary intervals in the data.

In many applications, you might want more granularity than just the total amount of a certain instance. For instance, if you data on event times, it could be very desirable to have the ability to query your data for the approximate amount an event occurs within arbtirary stretches of time (e.g. how many flows were there to a certain IP address from 10:00 to 10:15).

There is however a more creative use of these range queries -- approximating text relationships while being more storage efficient. We will be using this to make simple approximations of text similarity. This portion of the assignment is loosely inspired by [this paper](https://users.umiacs.umd.edu/~hal3//docs/daume10distsim.pdf).


In [ ]:
from google.colab import drive
from google.colab import files
import re
import math
import numpy as np
import mmh3
drive.mount('/content/drive/')
files.upload()


In [ ]:
FRANKPATH=NONE
with open(FRANKPATH,"r") as f:
    frankenstein = f.read()

In [ ]:

with open("Frankenstein.txt","r") as f:
    frankenstein = f.read()

# Hyperparameter: words per interval.
interval_words = 4000


# Basic cleaning: lowercase + punctuation removal + alphabetic-only tokenization.
clean_text = frankenstein.lower()
clean_text = re.sub(r"[^a-z\s]", " ", clean_text)
clean_text = clean_text.lower()
tokens = re.findall(r"[a-z]+", clean_text)


intervals = [
    tokens[i:i + interval_words]
    for i in range(0, len(tokens), interval_words)
]

print(f"Total words: {len(tokens):,}")
print(f"Interval size (words): {interval_words:,}")
print(f"Number of intervals: {len(intervals):,}")
print(f"First interval length: {len(intervals[0]):,}")
print(f"Last interval length: {len(intervals[-1]):,}")

class CountMinSketch:
    """
    Count-Min Sketch for approximate frequency estimation.

    Parameters
    ----------
    epsilon : float
        Additive error tolerance as fraction of total stream length N.
        Estimate ≤ true_count + epsilon * N  with prob ≥ 1 - delta.
    delta : float
        Failure probability. Smaller → more rows → more accurate.

    Methods
    -------
    update(x, count=1)  : Record `count` occurrences of item x.
    query(x)            : Return estimated frequency of x.
    merge(other)        : Merge another compatible CMS into this one (in-place).
    """

    def __init__(self, epsilon: float = 0.01, delta: float = 0.05):
        self.epsilon = epsilon
        self.delta   = delta
        self.w = math.ceil(math.e / epsilon)
        self.d = math.ceil(math.log(1.0 / delta))
        self.table = np.zeros((self.d, self.w), dtype=np.int64)
        self.seeds = list(range(self.d))
        self.total = 0                         # total events inserted

    # ── internals ──────────────────────────────────────────────────────────

    def _cols(self, x: str):
        """Return column index for each row."""
        key = str(x)
        return [mmh3.hash(key, seed=s, signed=False) % self.w
                for s in self.seeds]

    # ── public API ─────────────────────────────────────────────────────────

    def update(self, x, count: int = 1):
        """Insert item x with optional weight count (default 1)."""
        for i, col in enumerate(self._cols(x)):
            self.table[i, col] += count
        self.total += count

    def query(self, x) -> int:
        """Return estimated frequency of x (always ≥ true count)."""
        return int(min(self.table[i, col] for i, col in enumerate(self._cols(x))))

    def merge(self, other: "CountMinSketch"):
        """Element-wise add another CMS (same d, w). Useful for distributed sketches."""
        assert self.d == other.d and self.w == other.w, "Incompatible sketch dimensions"
        self.table += other.table
        self.total += other.total

    # ── diagnostics ────────────────────────────────────────────────────────

    def error_bound(self) -> float:
        """Theoretical maximum additive error: ε · N."""
        return self.epsilon * self.total

    def memory_bytes(self) -> int:
        return self.table.nbytes

    def __repr__(self):
        return (f"CountMinSketch(ε={self.epsilon}, δ={self.delta}, "
                f"d={self.d}, w={self.w:,}, "
                f"N={self.total:,}, mem={self.memory_bytes()/1024:.1f} KB)")


# Shared sketch accuracy hyperparameters.
epsilon = 0.005
delta = 0.01


##TODO: Create countmin sketches that correspond to fixed intervals of words in the book
interval_sketches = []
for interval in intervals:
    None


def query_interval(interval_idx: int, word: str) -> int:
    if interval_idx < 0 or interval_idx >= len(interval_sketches):
        raise IndexError("interval_idx out of range")
    return interval_sketches[interval_idx].query(word.lower())


def range_query(word: str, start_interval: int, end_interval: int) -> int:
  ##TODO: Define a function that naively returns the range query for given intervals
  #by utilizing the query_interval function.
    None



# Basic sanity check: common word should usually estimate higher than rare words.
for w in ["the", "monster"]:
    print(f"Total estimated count for '{w}': {range_query(w, 0, len(interval_sketches)-1)}")


#TODO: Run the following range queries:
#"victor" in the first 4000 words
#"fire" from the 12000th word to the 16000th word
#"science" for the first 20000 words

Does your range query retain the same level of approximate accuracy if you try to do a range query on 8000 words (starting at a correct split point)? Why or why not?

If you were trying to keep the errors for this range query bounded (with 99\% probability) to within a certain $\epsilon$, what would you have to do when constructing the intervals?

There's also another problem with this approach. In order for our approximation to hold at all, our range query needs to neatly conform to the intervals we've specified.

How can we make a series of count min sketches that can handle any arbitrary interval? Dyadic intervals!

In [ ]:
def build_dyadic_intervals(tokens, epsilon, delta):
    n = len(tokens)
    dyadic = {}  # key: (level, start_word) where block length = 2**level words

    if n == 0:
        return dyadic, n

    max_level = int(math.floor(math.log2(n)))
    for level in range(0, max_level + 1):
        block_len = 1 << level
        print("Level: "+str(level)+", Interval Lengths: "+str(block_len))
        for start_word in range(0, n, block_len):
            end_word = start_word + block_len
            if end_word > n:
                continue
            cms = CountMinSketch(epsilon=epsilon, delta=delta)
            for token in tokens[start_word:end_word]:
                cms.update(token)
            dyadic[(level, start_word)] = cms

    return dyadic, n



##TODO: Any arbitrary interval should be able to be represented as no more than
##2 log n dyadic ranges (where n is interval length).
#Choose an epsilon for the CMS in your dyadic intervals that, for any delta,
##should correspond to an epsilon of 0.05 on your range query.
RQEpsilon=None
dyadic_sketches, n_tokens = build_dyadic_intervals(
    tokens=tokens,
    epsilon=RQEpsilon,
    delta=delta,
)
#print(f"Total tokens (N): {n_tokens}")
#print(f"Dyadic blocks built from text: {len(dyadic_sketches)}")


def dyadic_range_query(word: str, start_word_idx: int, end_word_idx: int):
    if start_word_idx < 0 or end_word_idx >= n_tokens or start_word_idx > end_word_idx:
        raise ValueError("Invalid word-index range")

    i = start_word_idx
    segments = []  # (level, start_word, length)

    while i <= end_word_idx:
        max_k = int(math.floor(math.log2(end_word_idx - i + 1)))
        k = max_k

        # Find the largest aligned dyadic block starting at i.
        while k >= 0:
            block_len = 1 << k
            if (i % block_len == 0) and (i + block_len - 1 <= end_word_idx) and ((k, i) in dyadic_sketches):
                segments.append((k, i, block_len))
                i += block_len
                break
            k -= 1

        if k < 0:
            raise RuntimeError("Could not decompose range into dyadic blocks")

    estimate = sum(dyadic_sketches[(k, s)].query(word.lower()) for k, s, _ in segments)
    return {
        "word": word,
        "word_range": (start_word_idx, end_word_idx),
        "dyadic_segments": segments,
        "estimated_count": estimate,
    }


##TODO: Return the value of the range query for:
#Frankenstein, over the first 10000 words
#Victor from the 40000th to the 60347th word
#Creature from the 29th word to the 311th word
NFrankenstein=None
NVictor=None
NCreature=None

In [ ]:
# Estimate related-word frequency in [idx, idx + k] for every anchor-word position.

def estimate_from_anchor_positions(anchor_word: str, related_word: str, k: int = 20):
    if k < 0:
        raise ValueError("k must be non-negative")


    #TODO: Exact anchor positions from cleaned token stream.
    anchor_positions = None
    n_tokens=len(tokens)
    per_anchor_estimates = []
    for idx in anchor_positions:
        ##TODO: Use the dyadic_range query function on each individual appearance of the "anchor" word

        indiv_anchor_estimates.append(
            {
                "anchor_idx": None,
                "query_range": (None, None),
                "related_estimate": None,
            }
        )

    total_related_estimate = sum(item["related_estimate"] for item in indiv_anchor_estimates)


    return {
        "anchor_word": anchor_word,
        "related_word": related_word,
        "k": k,
        "num_anchor_positions": len(anchor_positions),
        "total_related_estimate_over_all_anchor_windows": total_related_estimate,
        "per_anchor_estimates": indiv_anchor_estimates,
    }


# Example queries.
ex1 = estimate_from_anchor_positions("frankenstein", "monster", k=20)
print({
    "anchor_word": ex1["anchor_word"],
    "related_word": ex1["related_word"],
    "k": ex1["k"],
    "num_anchor_positions": ex1["num_anchor_positions"],
    "anchors_with_nonzero_related_estimate": ex1["anchors_with_nonzero_related_estimate"],
    "total_related_estimate_over_all_anchor_windows": ex1["total_related_estimate_over_all_anchor_windows"],
})
print("First five per-anchor windows:", ex1["per_anchor_estimates"][:5])

ex2 = estimate_from_anchor_positions("victor", "creature", k=300)
print({
    "anchor_word": ex2["anchor_word"],
    "related_word": ex2["related_word"],
    "k": ex2["k"],
    "num_anchor_positions": ex2["num_anchor_positions"],
    "anchors_with_nonzero_related_estimate": ex2["anchors_with_nonzero_related_estimate"],
    "total_related_estimate_over_all_anchor_windows": ex2["total_related_estimate_over_all_anchor_windows"],
})

In [ ]:
##TODO: Find which words are the 50 most common in the data. Then, using the estimate_from_anchor_positions
##function, estimate the proportion of time other words in your list occur within 100 words of your "anchor word".
##Plot your results in a matrix with values ranging from 0 to 1.


import matplotlib.pyplot as plt



Your plot should be mostly prepositional words. A good sanity check but perhaps not very informative in terms of identifying relationships between words. What happens if we use a random selection of words from the book?

In [ ]:
##TODO: Generate a similar picture to the last section but now using 70 random words
##that each appear in the book more than eight times. Most cells will be zero, but for some pairs of words you should be able to see
##hints of a relationship.
rng = np.random.default_rng(808)


# Export Your Results to JSON File

In [ ]:
import json
json_output_path = "TextRangeQueries.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump({
        "NFrankenstein":NFrankenstein,
        "NVictor": NVictor,
        "NCreature": NCreature,
        "RQEpsilon":RQEpsilon,
    }, f, indent=2)